In [3]:
import gzip
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import glob

In [ ]:
def load_json_gzp(path: str) -> pd.DataFrame:
    records = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

STATE_NAME = "Virginia"
meta_path = f"data/meta-{STATE_NAME}.json.gz" 
review_path = f"data/review-{STATE_NAME}.json.gz" 

df_meta = load_json_gzp(meta_path)
df_review = load_json_gzp(review_path)

Keep only business id, avg rating, and total # of reviews. Also drops rows with missing values

In [5]:
biz = df_meta[["gmap_id", "avg_rating", "num_of_reviews"]].dropna()

biz["num_of_reviews"] = biz["num_of_reviews"].astype(int)
biz["avg_rating"] = biz["avg_rating"].astype(float)


In [6]:
#X-axis labels and bins
bins = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, biz["num_of_reviews"].max() + 1]
labels = [f"{bins[i]}–{bins[i+1]-1}" for i in range(len(bins)-1)]

biz["review_bin"] = pd.cut(biz["num_of_reviews"], bins=bins, labels=labels, right=False)

#Calculate statistics for each bin
stability_table = (biz.groupby("review_bin").agg(
        n_businesses=("gmap_id", "count"),
        rating_std=("avg_rating", "std"),
        rating_var=("avg_rating", "var"),
        mean_rating=("avg_rating", "mean"),
    ).reset_index())

#Is this alright?
print(stability_table)

  review_bin  n_businesses  rating_std  rating_var  mean_rating
0        1–1          4228    1.223072    1.495906     4.271949
1        2–4         10213    0.875691    0.766835     4.282415
2        5–9         19405    0.722017    0.521309     4.263679
3      10–19         15062    0.661144    0.437111     4.256619
4      20–49         21531    0.620459    0.384970     4.268069
5      50–99         16440    0.567684    0.322265     4.286089
6    100–199         12988    0.492077    0.242140     4.310371
7    200–499         12316    0.451288    0.203661     4.282941
8    500–999          5013    0.410508    0.168517     4.214981
9  1000–9998          2477    0.381416    0.145479     4.251756


C:\Users\aiden\AppData\Local\Temp\ipykernel_12816\4020917387.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stability_table = (biz.groupby("review_bin").agg(


In [7]:
#Change in std between bins
stability_table["delta_std"] = stability_table["rating_std"].diff().abs()
stability_table

epsilon = 0.02  # ratings barely change beyond this
stable_bins = stability_table[stability_table["delta_std"] < epsilon]
#Weird output
print(stable_bins)


Empty DataFrame
Columns: [review_bin, n_businesses, rating_std, rating_var, mean_rating, delta_std]
Index: []


In [8]:
#40+ reviews is stable (good marker?)
biz_stable = biz[biz["num_of_reviews"] >= 40]

In [9]:
#Scale big numbers down because comparing was hard
biz["weight"] = np.log1p(biz["num_of_reviews"])

In [10]:
#Makes grid over VA. Gemini helped heavily with this
def add_geo_cells(df, lat_col="latitude", lon_col="longitude", cell_size=0.1):
    #get used to using copy more and understanding it
    d = df.copy()
    d["lat_cell"] = (d[lat_col] // cell_size) * cell_size
    d["lon_cell"] = (d[lon_col] // cell_size) * cell_size
    d["geo_cell"] = d["lat_cell"].astype(str) + "_" + d["lon_cell"].astype(str)
    return d

df_meta_cells = add_geo_cells(df_meta)


In [11]:
#Calculate statistics for each "grid"
cell_features = (
    df_meta_cells.groupby("geo_cell").agg(
        n_businesses=("gmap_id", "nunique"),
        avg_reviews_per_biz=("num_of_reviews", "mean"),
        median_reviews_per_biz=("num_of_reviews", "median"),
        pct_high_review_biz=("num_of_reviews", lambda x: (x >= 100).mean()),
        avg_rating=("avg_rating", "mean"),
    ).reset_index())


In [12]:
if "categories" in df_meta_cells.columns:
    cat_div = (
        df_meta_cells.explode("categories")
        .groupby("geo_cell")["categories"]
        .nunique()
        .rename("category_diversity")
    )
    #Come back to this
    cell_features = cell_features.merge(cat_div, on="geo_cell", how="left")

In [13]:
#density metrics
features = ["n_businesses", "avg_reviews_per_biz", "pct_high_review_biz",]

#Why this again?
if "category_diversity" in cell_features.columns:
    features.append("category_diversity")

X = cell_features[features].fillna(0)

#Look more into this
#z-scores Ex: n_businesses = 500 but pct_high_review_biz = 0.25
scaler = StandardScaler()
#turn raw nums into standardized z-scores and get mean of all scores of each cell
cell_features["urban_index"] = scaler.fit_transform(X).mean(axis=1)

In [14]:
#split based on biz density
cell_features["urban_rural"] = pd.qcut(cell_features["urban_index"], q=2, labels=["rural", "urban"])

In [15]:
#give each biz label
df_meta_cells = df_meta_cells.merge(cell_features[["geo_cell", "urban_rural", "urban_index"]], on="geo_cell", how="left")

#same thing
df_review = df_review.merge(df_meta_cells[["gmap_id", "urban_rural", "urban_index"]], on="gmap_id", how="left")


In [16]:
df_meta_cells.groupby("urban_rural").agg(
    avg_rating=("avg_rating", "mean"),
    median_reviews=("num_of_reviews", "median"),
    pct_5star=("avg_rating", lambda x: (x >= 4.5).mean()), #review with Dhruv or Ansh
)


C:\Users\aiden\AppData\Local\Temp\ipykernel_12816\3960716687.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_meta_cells.groupby("urban_rural").agg(


,avg_rating,median_reviews,pct_5star
urban_rural,,,
rural,4.489671,8.0,0.653840
urban,4.262043,38.0,0.462294


In [17]:
#Too many empty text on reviews
df_review["review_len"] = df_review["text"].fillna("").str.len()

df_review.groupby("urban_rural").agg(
    avg_review_rating=("rating", "mean"),
    avg_review_length=("review_len", "mean"),
    n_reviews=("gmap_id", "size"),
)

C:\Users\aiden\AppData\Local\Temp\ipykernel_12816\4140258023.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_review.groupby("urban_rural").agg(


,avg_review_rating,avg_review_length,n_reviews
urban_rural,,,
rural,4.489554,98.592364,165086
urban,4.262017,99.413384,15816382


In [18]:
#groups bizs into bins and calculates density metrics
def extract_cell_features(df):
    d = df.copy()
    
    #creates grid
    CELL_SIZE = 0.1 
    d["lat_cell"] = (d["latitude"] // CELL_SIZE) * CELL_SIZE
    d["lon_cell"] = (d["longitude"] // CELL_SIZE) * CELL_SIZE
    d["geo_cell"] = d["lat_cell"].astype(str) + "_" + d["lon_cell"].astype(str)
    
    #calculate features per cell
    features = d.groupby("geo_cell").agg(
        n_businesses=("gmap_id", "nunique"),
        avg_reviews_per_biz=("num_of_reviews", "mean"),
        pct_high_review_biz=("num_of_reviews", lambda x: (x >= 100).mean())
    )
    
    return features, d

In [19]:
#finds ever file in data
all_files = glob.glob("data/meta-*.json.gz")
global_feature_list = []

for file in all_files:
    temp_df = load_json_gzp(file)[["gmap_id", "num_of_reviews", "latitude", "longitude"]].dropna()
    feats, _ = extract_cell_features(temp_df)
    global_feature_list.append(feats)

#combine cells
combined_cells = pd.concat(global_feature_list)
scaler = StandardScaler().fit(combined_cells)

#Calculate threshholds
global_index = scaler.transform(combined_cells).mean(axis=1)
low_threshold = np.percentile(global_index, 33.3)  # Rural ends here
high_threshold = np.percentile(global_index, 66.6) # Suburban ends here

In [21]:
feats, df_with_cells = extract_cell_features(df_meta)

#scale state using global scalar
# Wrap state_scores in a pd.Series
state_scores = pd.Series(scaler.transform(feats).mean(axis=1), index=feats.index)


#apply label based on threshold
def classify_region(val):
    if val < low_threshold: return "rural"
    if val < high_threshold: return "suburban"
    return "urban"

feats["region_type"] = state_scores.apply(classify_region)

#merge back to the metadata
df_meta = df_with_cells.merge(feats[["region_type"]], on="geo_cell", how="left")

In [22]:
#Had to readd due to missing column. Look back in code to merge later
bins = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, df_meta["num_of_reviews"].max() + 1]
labels = [f"{bins[i]}–{bins[i+1]-1}" for i in range(len(bins)-1)]

df_meta["review_bin"] = pd.cut(df_meta["num_of_reviews"], bins=bins, labels=labels, right=False)

regional_summary = df_meta.groupby(["region_type", "review_bin"]).agg(
    rating_std=("avg_rating", "std"), 
    n_businesses=("gmap_id", "count")
).reset_index()

C:\Users\aiden\AppData\Local\Temp\ipykernel_12816\2597214010.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  regional_summary = df_meta.groupby(["region_type", "review_bin"]).agg(


In [23]:
#grouping for tableau
regional_summary = df_meta.groupby(["region_type", "review_bin"]).agg(
    rating_std=("avg_rating", "std"), 
    n_businesses=("gmap_id", "count")
).reset_index()

C:\Users\aiden\AppData\Local\Temp\ipykernel_12816\555981880.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  regional_summary = df_meta.groupby(["region_type", "review_bin"]).agg(


In [24]:
#Which states are you using?
print(all_files)

['data\\meta-District_of_Columbia.json.gz', 'data\\meta-Virginia.json.gz', 'data\\meta-Wyoming.json.gz']


In [25]:
regional_summary

,region_type,review_bin,rating_std,n_businesses
0,rural,1–1,1.013949,141
1,rural,2–4,0.747182,260
2,rural,5–9,0.425686,271
3,rural,10–19,0.521950,82
4,rural,20–49,0.362589,49
5,rural,50–99,0.228035,6
6,rural,100–199,NaN,0
7,rural,200–499,NaN,0
8,rural,500–999,NaN,0
9,rural,1000–9998,NaN,0


In [ ]:
#Don't forget to run this
%pip install plotly nbformat

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.9 MB 7.7 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.9 MB 11.6 MB/s eta 0:00:01
   -------------------------------- ------- 8.1/9.9 MB 13.3 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 12.8 MB/s  0:00:00

   ---------------------------------------- 0/4 [fastjsonschema]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import plotly.express as px

#Drop blank biz values
plot_data = regional_summary.dropna(subset=['rating_std'])

fig = px.line(
    plot_data, 
    x="review_bin", 
    y="rating_std", 
    color="region_type",
    markers=True,
    title="Rating Stability (Std Dev) by Review Volume and Region",
    labels={"rating_std": "Rating Standard Deviation", "review_bin": "Number of Reviews"},
    category_orders={"review_bin": labels} # Ensures bins stay in numerical order
)

fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()

In [ ]:
#biz summary but added suburban
meta_summary = df_meta.groupby("region_type").agg(
    avg_rating=("avg_rating", "mean"),
    median_reviews=("num_of_reviews", "median"),
    pct_high_rated=("avg_rating", lambda x: (x >= 4.5).mean()),
    total_businesses=("gmap_id", "count")
).reindex(["rural", "suburban", "urban"]) # Keeps them in logical order

print("--- Metadata Summary (Business Level) ---")
print(meta_summary)

--- Metadata Summary (Business Level) ---
             avg_rating  median_reviews  pct_high_rated  total_businesses
region_type                                                              
rural          4.514339             5.0        0.682324               809
suburban       4.480057            13.0        0.643746              6995
urban          4.258643            38.0        0.459502            111869


In [ ]:
#make sure df_review has new region_type from df_meta
if 'region_type' not in df_review.columns:
    df_review = df_review.merge(df_meta[['gmap_id', 'region_type']], on='gmap_id', how='left')

df_review["review_len"] = df_review["text"].fillna("").str.len()

#user summary but added suburban
review_summary = df_review.groupby("region_type").agg(
    avg_review_rating=("rating", "mean"),
    avg_review_length=("review_len", "mean"),
    n_reviews=("gmap_id", "size"),
).reindex(["rural", "suburban", "urban"])

print("\n--- Review Summary (User Behavior) ---")
print(review_summary)


--- Review Summary (User Behavior) ---
             avg_review_rating  avg_review_length  n_reviews
region_type                                                 
rural                 4.532718          88.724652       6174
suburban              4.469021          94.772991     255214
urban                 4.261284          99.761850   15767140
